In [ ]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
import warnings

It is old version of training notebook. The new one is trani_corner_models_evaluate.ipynb and it is highly recommended to use it instead of this one.

In [ ]:
esa_21_22_corners_parsed = pd.read_pickle('data/corners_xgb_esa_21_22_0912.pkl')
esa_22_23_corners_parsed = pd.read_pickle('data/corners_xgb_esa_22_23_0912.pkl')
euro_corners_parsed = pd.read_pickle('data/corners_xgb_euro_0912.pkl')
fifa_corners_parsed = pd.read_pickle('data/corners_xgb_fifa_2022_0912.pkl')

esa_21_22_core = pd.read_pickle('data/38_108.pkl')
esa_22_23_core = pd.read_pickle('data/38_235.pkl')

In [ ]:
# label = 'was_shot'
label = 'shot_on_goal'

In [ ]:
def get_test_set(df, df_core):
    dfs_test = []
    dfs_train = []
    teams = list(set(df['team'].values))
    test_matches = df_core[esa_22_23_core['match_week'] >= 18]['match_id'].values
    train_matches = df_core[esa_22_23_core['match_week'] < 18]['match_id'].values
    for team in teams:
        for match_id in test_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_test.append(df_result)
        for match_id in train_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_train.append(df_result)
    df_test = pd.concat(dfs_test)
    df_train = pd.concat(dfs_train)
    return df_test, df_train


esa_22_23_test, esa_22_23_train = get_test_set(esa_22_23_corners_parsed, esa_22_23_core)
print(f'Length of 22/23 test set: {len(esa_22_23_test)}')
print(f'Length of 22/23 train set: {len(esa_22_23_train)}')

In [ ]:
test_set = esa_22_23_test
train_set = pd.concat([esa_22_23_train, esa_21_22_corners_parsed, euro_corners_parsed, fifa_corners_parsed])

print(f'Length of test set: {len(test_set)}')
print(f'Length of train set: {len(train_set)}')

print(f'Length of esa_22_23_test: {len(esa_22_23_test)}')
print(f'Length of esa_22_23_train: {len(esa_22_23_train)}')
print(f'Length of esa_21_22_corners_parsed: {len(esa_21_22_corners_parsed)}')
print(f'Length of euro_corners_parsed: {len(euro_corners_parsed)}')
print(f'Length of fifa_corners_parsed: {len(fifa_corners_parsed)}')


In [ ]:
test_set.columns

In [ ]:
columns =['match_id', 'id', 'team', 'num_of_events', 'duration', 'side_right',
       'side_left', 'first_event_end_x', 'first_event_end_y', 'num_of_pass',
       'num_of_carry', 'num_of_miscontrol', 'num_of_dribble', 'num_of_error',
       'num_of_duel', 'num_of_ball_receipt*', 'num_of_clearance',
       'num_of_ball_recovery', 'num_of_dispossessed', 'num_of_interception',
       'num_of_block', 'num_of_shot', 'num_of_foul_committed',
       'num_of_foul_won', 'corner_score', 'obv', 'xg', 'num_of_passes_to_I',
       'num_of_passes_to_II', 'num_of_passes_to_LZ', 'num_of_passes_to_RZ',
       'num_of_passes_to_LO', 'num_of_passes_to_CO', 'num_of_passes_to_RO',
       'num_of_passes_to_LP', 'num_of_passes_to_RP', 'num_of_passes_to_CP',
       'shot_on_goal']



In [ ]:
train_set.columns

In [ ]:
def prepare_df_with_label(train_set, test_set, label):
    if label == 'was_shot':
        columns_to_drop = ['id','match_id', 'team', 'corner_score', 'obv', 'xg', 'shot_on_goal', 'num_of_shot']
        test_set['label'] = [1 if (shots > 0) else 0 for shots in test_set['num_of_shot'].values]
        train_set['label'] = [1 if (shots > 0) else 0 for shots in train_set['num_of_shot'].values]

    elif label == 'shot_on_goal':
        columns_to_drop = ['id','team', 'corner_score', 'obv', 'xg', 'num_of_shot', 'shot_on_goal', 'match_id']
        test_set['label'] = [1 if (shots > 0) else 0 for shots in test_set['shot_on_goal'].values]
        train_set['label'] = [1 if (shots > 0) else 0 for shots in train_set['shot_on_goal'].values]

    else:
        raise ValueError('Incorrect name of label!')
    
    print(f'Number of {label} in test set: {sum(test_set["label"].values)}')
    print(f'Number of {label} in train set: {sum(train_set["label"].values)}')

    num_of_positive = sum(test_set["label"].values) +sum(train_set["label"].values)
    num_of_negative = len(test_set) + len(train_set) - num_of_positive

    print(f'Number of positive in whole set: {num_of_positive}')
    print(f'Number of negative in whole set: {num_of_negative}')

    data_balance = num_of_negative / num_of_positive
    print(f'Data balance: {data_balance}')

    return train_set, test_set, columns_to_drop, data_balance

In [ ]:
train_set, test_set, columns_to_drop, data_balance = prepare_df_with_label(train_set, test_set, label)

In [ ]:
def get_sets_for_model(df_train, df_test):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    y_train = df_train['label']
    X_train = df_train.drop(columns=['label'])

    y_test = df_test['label']
    X_test = df_test.drop(columns=['label'])

    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set)

In [ ]:
X_test.columns

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split


def train_model(X_train, X_test, y_train, columns_to_drop, data_balance):
    # parameters =  {'colsample_bytree': 0.8, 'learning_rate': 0.3, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.8}
    
    # best parameters for shot_on_goal
    parameters = {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8}
    
    
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance, **parameters)
    # model = XGBClassifier(objective='binary:logistic')
    input_X_train = X_train.drop(columns=columns_to_drop)
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=columns_to_drop)
    XGB_predictions = model.predict(input_X_test)
#######


    # param_grid = {
    #     'scale_pos_weight': [0.2, 1, 4],
    #     'n_estimators': [100, 200, 300],
    #     'max_depth': [3, 6],
    #     'learning_rate': [0.01, 0.1, 0.3],
    #     'subsample': [0.8, 1.0],
    #     'colsample_bytree': [0.8, 1.0]
    # }
    # param_grid = {
    #     'scale_pos_weight': [4, 4.7, 5.5],
    #     'n_estimators': [300, 400],
    #     'max_depth': [2, 3, 4],
    #     'learning_rate': [0.1],
    #     'subsample': [0.8],
    #     'colsample_bytree': [1.0]
    # }
    # grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='f1', cv=3, verbose=1)
    # grid_search.fit(input_X_train, y_train)
    # print("Najlepsze parametry:", grid_search.best_params_)
######    
    
    
    return model, XGB_predictions, input_X_test

model, pred, input_test = train_model(X_train, X_test, y_train, columns_to_drop, data_balance)

In [ ]:
X_test.columns

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(y_test, pred):
    print('Confusion matrix')
    print(confusion_matrix(y_test, pred))
    print('Classification report')
    print(classification_report(y_test, pred))
    print('Accuracy= ', accuracy_score(y_test, pred))

evaluate_model(y_test, pred)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(accuracy_score(y_test, pred))
print(precision_score(y_test, pred))
print(recall_score(y_test, pred))
print(f1_score(y_test, pred))

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.1, 0.2, 0.3],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='f1', cv=3, verbose=1)
grid_search.fit(X_train, y_train)
print("Najlepsze parametry:", grid_search.best_params_)


In [ ]:
for data_balance in range(0, 10):
    data_balance = data_balance / 10
    print('==============')
    print(data_balance)
    model, pred, input_test = train_model(X_train, X_test, y_train, columns_to_drop, data_balance)
    evaluate_model(y_test, pred)

In [ ]:
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()
 
print("now =", now)

# dd/mm/YY H:M:S
dt_string = now.strftime("%d%m%Y_%H%M%S")
print("date and time =", dt_string)

In [ ]:
def feature_importances_raport(input_X_test, model, set_title):
    result = pd.DataFrame(columns=['name', 'value'])
    result['name'] = input_X_test.columns
    result['value'] = model.feature_importances_

    result = result.sort_values(by=['value'], ascending=False)

    result.to_csv(f'features_{set_title}.csv')
    print(result)
    return result

feature_importances = feature_importances_raport(input_test, model, f'corners_experiment_{dt_string}')

In [ ]:
def generate_scores(model, X_test, X_train, y_train, set_title, columns_to_drop, data_balance):
    # model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance)
    model = XGBClassifier(objective='binary:logistic')
    input_X_train = X_train.drop(columns=columns_to_drop)
    
    model.fit(input_X_train, y_train)
    input_X_test = X_test.drop(columns=columns_to_drop)

    model = model.predict_proba(input_X_test)

    X_test['score_positive'] = [m[1] for m in model]
    X_test['score_negative'] = [m[0] for m in model]

    X_test.to_csv(f'{set_title}_evaluated.csv')

    return X_test

df_with_scores = generate_scores(model, X_test, X_train, y_train, f'corners_experiment_{dt_string}', columns_to_drop, data_balance)

In [ ]:
test_set.columns

In [ ]:
df_with_scores.columns

In [ ]:
df_with_scores.to_csv('corners_with_scores_example.csv')

In [ ]:
df_with_scores.to_pickle('corners_with_scores_example_final.pkl')

In [ ]:
def show_correlation(df_with_scores, test_set):
    # df_with_scores = df_with_scores.merge(test_set[['id','xg']], on='id')
    # df_with_scores = df_with_scores.merge(test_set[['id','obv']], on='id')
    # df_with_scores = df_with_scores.merge(test_set[['id','corner_score']], on='id')
    corr_with_xg = df_with_scores['score_positive'].corr(df_with_scores['xg'])
    corr_with_obv = df_with_scores['score_positive'].corr(df_with_scores['obv'])
    corr_with_corner_score = df_with_scores['score_positive'].corr(df_with_scores['corner_score'])
    print(f'Correlation with xg: {corr_with_xg:.3f}')
    print(f'Correlation with obv: {corr_with_obv:.3f}')
    print(f'Correlation with corner_score: {corr_with_corner_score:.3f}')
    return df_with_scores

In [ ]:
df_with_scores.columns

In [ ]:
test_set.columns

In [ ]:
df_with_scores.columns

In [ ]:
df_with_scores = show_correlation(df_with_scores, test_set)

**Corners: default columns**\
Correlation with xg: 0.462\
Correlation with obv: 0.150\
Correlation with corner_score: 0.341


**Corners: columns with num to every zone passes**\
Correlation with xg: 0.463\
Correlation with obv: 0.192\
Correlation with corner_score: 0.345


**Corners: columns with num to every zone passes and label is shot on goal**\
Correlation with xg: 0.469\
Correlation with obv: 0.156\
Correlation with corner_score: 0.268


**Corners: correlation final**\
Correlation with xg: 0.345\
Correlation with obv: 0.103\
Correlation with corner_score: 0.282

In [ ]:
def get_avg_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'score_positive': 'mean', 'obv': 'mean', 'xg': 'mean', 'corner_score' : 'mean', 'num_of_shot' : 'mean', 'shot_on_goal' : 'mean'}).reset_index()
    df_with_scores['total_corners'] = df_with_scores['team'].map(test_set['team'].value_counts()) / 18
    df_with_scores = df_with_scores.sort_values(by=['score_positive'], ascending=False)
    return df_with_scores.round(3)

def get_sum_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'score_positive': 'sum', 'obv': 'sum', 'xg': 'sum', 'corner_score' : 'sum', 'num_of_shot' : 'sum', 'shot_on_goal' : 'sum'}).reset_index()
    df_with_scores['total_corners'] = df_with_scores['team'].map(test_set['team'].value_counts())
    df_with_scores = df_with_scores.sort_values(by=['score_positive'], ascending=False)
    return df_with_scores.round(3)

def get_med_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'score_positive': 'median', 'obv': 'median', 'xg': 'median', 'corner_score' : 'median'}).reset_index()
    df_with_scores = df_with_scores.sort_values(by=['score_positive'], ascending=False)
    return df_with_scores.round(3)

In [ ]:
df_with_scores

In [ ]:
res_avg = get_avg_metrics_for_each_team(df_with_scores, test_set)
res_sum = get_sum_metrics_for_each_team(df_with_scores, test_set)
res_med = get_med_metrics_for_each_team(df_with_scores, test_set)

In [ ]:
res_sum

In [ ]:
res_avg

In [ ]:
res_sum.to_pickle('corners_with_scores_example_final_sum.pkl')
res_avg.to_pickle('corners_with_scores_example_final_avg.pkl')

In [ ]:
import pandas as pd


In [ ]:
import shap
import xgboost as xgb

def generate_feature_importances(model, X_test, X_train, y_train, columns_to_drop, data_balance):
    parameters = {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8}

    # model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance)
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance, **parameters)
    input_X_train = X_train.drop(columns=columns_to_drop)
    
    model.fit(input_X_train, y_train)
    input_X_test = X_test.drop(columns=columns_to_drop)

    explainer = shap.TreeExplainer(model)
    column_to_group_by = 'team'
    groups = X_test.groupby(column_to_group_by)
    features_importance_per_group = {}

    data_whole = xgb.DMatrix(input_X_test)
    shap_values_whole = explainer.shap_values(data_whole)
    mean_importance_whole = abs(shap_values_whole).mean(axis=0)
    column_indices_whole = input_X_test.columns
    feature_names_whole = list(column_indices_whole)
    features_importance_per_group['whole_test_set'] = {'importance': mean_importance_whole, 'feature_names': feature_names_whole}

    
    for group_name, group_data in groups:
        data = xgb.DMatrix(group_data.drop(columns=columns_to_drop))
        shap_values = explainer.shap_values(data)
        mean_importance = abs(shap_values).mean(axis=0)
        # features_importance_per_group[group_name] = mean_importance

        column_indices = group_data.drop(columns=columns_to_drop).columns
        feature_names = list(column_indices)
        features_importance_per_group[group_name] = {'importance': mean_importance, 'feature_names': feature_names}

    return features_importance_per_group

importances_per_team = generate_feature_importances(model, X_test, X_train, y_train, columns_to_drop, data_balance)

In [ ]:
importances_per_team['whole_test_set']

In [ ]:
importances_per_team['Lech Poznan']

In [ ]:
import pandas as pd

# Wczytaj zbiór testowy
df = pd.read_csv('test_data.csv')  # Załóżmy, że dane są w formacie CSV

# Przykładowa kolumna do podziału na podgrupy
column_to_group_by = 'kategoria'

# Podziel zbiór testowy na podgrupy na podstawie wybranej kolumny
grupy = df.groupby(column_to_group_by)


In [ ]:
import shap
import xgboost as xgb

# Wczytaj wytrenowany model XGBoost
model = xgb.Booster()
model.load_model('model.model')  # Załaduj wytrenowany model

# Inicjalizuj eksplainera SHAP dla danego modelu
explainer = shap.TreeExplainer(model)

# Inicjalizuj słownik do przechowywania ważności cech dla każdej podgrupy
features_importance_per_group = {}

# Oblicz SHAP values i ważność cech dla każdej podgrupy
for group_name, group_data in grupy:
    data = xgb.DMatrix(group_data.drop(columns=[column_to_group_by]))  # Usuń kolumnę 'kategoria'
    shap_values = explainer.shap_values(data)
    
    # Oblicz średnią ważność cech dla danej podgrupy
    mean_importance = abs(shap_values).mean(axis=0)
    
    # Zapisz ważność cech dla danej podgrupy w słowniku
    features_importance_per_group[group_name] = mean_importance
